# Content Refresh Prioritization Model

**FlyRank Internship Project**

**Task:** Given a dataset of ~30k web pages with search performance data (rankings, clicks, content signals, engagement, etc.), predict which pages should be prioritized for a content refresh. This is framed as a binary classification problem: does a given page currently *need* a refresh?

**Approach**
1. Clean and prepare the raw data (missing values, data types, unit inconsistencies, outliers).
2. Engineer features that capture traffic momentum, engagement, content staleness, and search opportunity.
3. Define a refresh-need label from domain logic (a page is stale, declining, *and* poorly ranked).
4. Build a leakage-safe feature matrix and train baseline classifiers.
5. Tune the decision threshold and evaluate with cross-validation.


## 1. Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", 100)

## 2. Load Data

In [ ]:
df = pd.read_csv("content_refresh_anonymized.csv")
df.shape

## 3. Initial Exploration

Quick look at the shape, structure, and quality of the data before doing any cleaning.

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df.duplicated().sum()

In [ ]:
df.isnull().sum()[df.isnull().sum() > 0]

## 4. Handling Missing Values

Missing values fall into a few natural groups. Each group is handled with a rule that matches
what the missingness actually means, rather than a single blanket strategy:

- **AI generation metadata** (`provider_used`, `model_used`): missing means the content was not
  AI-generated, so it's filled with an explicit `"not_ai_generated"` label rather than dropped.
- **Keyword / search metrics** (`search_volume`, `cpc`, `competition`, `competition_level`):
  median-filled for numeric columns, with a flag column to preserve the fact that the value was
  originally missing.
- **Content structure** (`word_count`, `char_count` and their tiers): median/`"unknown"`-filled,
  again with a missingness flag for `word_count`.
- **Intent** (`main_intent`): filled with `"unknown"`.
- **Scroll rate**: small numeric gap, safe to median-fill.
- **Trend percentage** (`trend_pct`): checked against prior-period traffic first — rows with
  missing trend mostly have zero prior impressions, meaning there's no trend to measure, so these
  are filled with `0` rather than a median.

In [ ]:
# --- AI generation metadata ---
df["provider_used"] = df["provider_used"].fillna("not_ai_generated")
df["model_used"] = df["model_used"].fillna("not_ai_generated")

# --- Keyword / search metrics ---
df["kw_data_missing"] = df["search_volume"].isnull().astype(int)
df["search_volume"] = df["search_volume"].fillna(df["search_volume"].median())
df["cpc"] = df["cpc"].fillna(df["cpc"].median())
df["competition"] = df["competition"].fillna(df["competition"].median())
df["competition_level"] = df["competition_level"].fillna("unknown")

# --- Content structure ---
df["content_not_scraped"] = df["word_count"].isnull().astype(int)
df["word_count"] = df["word_count"].fillna(df["word_count"].median())
df["char_count"] = df["char_count"].fillna(df["char_count"].median())
df["word_count_tier"] = df["word_count_tier"].fillna("unknown")
df["char_count_tier"] = df["char_count_tier"].fillna("unknown")

# --- Intent ---
df["main_intent"] = df["main_intent"].fillna("unknown")

# --- Scroll rate ---
df["scroll_rate"] = df["scroll_rate"].fillna(df["scroll_rate"].median())

# --- Trend percentage ---
# Missing trend_pct rows mostly have zero prior impressions (no trend to measure), so fill with 0
no_prior_traffic_check = df[df["trend_pct"].isnull()][["impressions_prev_30d", "impressions_last_30d"]].describe()
print(no_prior_traffic_check)
df["trend_pct"] = df["trend_pct"].fillna(0)

In [ ]:
# Confirm no missing values remain
df.isnull().sum()[df.isnull().sum() > 0]

## 5. Type Casting

Cast the low-cardinality string columns to `category` dtype for clarity and memory efficiency.

In [ ]:
categorical_cols = [
    "content_type", "main_intent", "provider_used", "model_used",
    "competition_level", "age_tier", "freshness_tier",
    "word_count_tier", "char_count_tier", "impression_tier",
    "position_tier", "trend_direction",
]
for col in categorical_cols:
    if col in df.columns:
        df[col] = df[col].astype("category")

df.head(3)

## 6. Fixing a Unit Inconsistency in CTR

`ctr`, `engagement_rate`, and `ai_traffic_pct` are expected to be fractions between 0 and 1.
A quick check shows some `ctr` values well above 1, which turns out to be a units problem —
a subset of rows were recorded as percentages (0–100) instead of fractions (0–1).

In [ ]:
df[["ctr", "engagement_rate", "ai_traffic_pct"]].describe()

In [ ]:
print("ctr values > 1:", (df["ctr"] > 1).sum())
df.loc[df["ctr"] > 1, "ctr"].describe()

In [ ]:
# Rows recorded on a 0-100 scale get rescaled down to the 0-1 fraction the rest of the column uses
df.loc[df["ctr"] > 1, "ctr"] = df.loc[df["ctr"] > 1, "ctr"] / 100

print("ctr values > 1 after fix:", (df["ctr"] > 1).sum())
df["ctr"].describe()

### Domain-rule outlier check

With `ctr` now on a consistent 0–1 scale, check the rest of the columns against basic
domain rules (impossible or nonsensical values) rather than a purely statistical outlier method —
this data is skewed enough that IQR/Z-score rules would flag a lot of legitimate high-traffic pages.

In [ ]:
domain_violations = (
    (df["avg_position"] <= 0) |
    (df["ctr"] > 1) | (df["ctr"] < 0) |
    (df["engagement_rate"] > 1) | (df["engagement_rate"] < 0) |
    (df["ai_traffic_pct"] > 1) | (df["ai_traffic_pct"] < 0) |
    (df["word_count"] <= 0) |
    (df["clicks_90d"] < 0) | (df["impressions_90d"] < 0)
)

print("Domain rule violations:", domain_violations.sum())
df[domain_violations].head()

## 7. Feature Engineering

New features are grouped by what they measure: traffic momentum, activity coverage, engagement,
AI-driven traffic, content staleness, search opportunity, and content thinness.

### Traffic momentum
Percent change in clicks, impressions, and sessions between the last 30 days and the prior 30 days.
Ratios are clipped to `[-1, 3]` to avoid a handful of near-zero denominators producing extreme values.

In [ ]:
df["traffic_change_pct"] = (
    (df["clicks_last_30d"] - df["clicks_prev_30d"]) / df["clicks_prev_30d"].replace(0, np.nan)
).fillna(0).clip(-1, 3)

df["impressions_change_pct"] = (
    (df["impressions_last_30d"] - df["impressions_prev_30d"]) / df["impressions_prev_30d"].replace(0, np.nan)
).fillna(0).clip(-1, 3)

df["sessions_change_pct"] = (
    (df["sessions_last_30d"] - df["sessions_prev_30d"]) / df["sessions_prev_30d"].replace(0, np.nan)
).fillna(0).clip(-1, 3)

### Activity coverage
How consistently a page receives clicks/impressions, and what share of the 90-day window it was active.

In [ ]:
df["clicks_per_active_day"] = (df["clicks_90d"] / df["days_with_sessions"].replace(0, np.nan)).fillna(0)
df["impressions_per_active_day"] = (df["impressions_90d"] / df["days_with_impressions"].replace(0, np.nan)).fillna(0)

df["impression_coverage"] = df["days_with_impressions"] / 90
df["session_coverage"] = df["days_with_sessions"] / 90

### Engagement
Session-level engagement signals normalized per session or per user.

In [ ]:
df["engaged_session_rate"] = (df["engaged_sessions_90d"] / df["sessions_90d"].replace(0, np.nan)).fillna(0)
df["scroll_events_per_session"] = (df["scroll_events_90d"] / df["sessions_90d"].replace(0, np.nan)).fillna(0)
df["pageviews_per_user"] = (df["pageviews_90d"] / df["users_90d"].replace(0, np.nan)).fillna(0)

### AI-driven traffic share
Computed independently from session-level counts as a sanity check against the pre-existing
`ai_traffic_pct` column, then clipped to a valid `[0, 1]` range.

In [ ]:
df["ai_session_share"] = (df["ai_sessions_90d"] / df["sessions_90d"].replace(0, np.nan)).fillna(0)
df["ai_session_share"] = df["ai_session_share"].clip(upper=1.0)

# Sanity check: should closely match the existing ai_traffic_pct column
(df["ai_session_share"] - df["ai_traffic_pct"]).abs().describe()

### Content staleness
A page is flagged as "stale and declining" if it hasn't been updated in longer than the median
gap *and* its traffic trend is explicitly downward. `update_lag_ratio` measures how much of a
page's lifetime has passed since its last meaningful update (close to 1 = never really refreshed,
close to 0 = recently refreshed).

In [ ]:
df["stale_and_declining"] = (
    (df["days_since_last_update"] > df["days_since_last_update"].median()) &
    (df["trend_direction"] == "down")
).astype(int)

df["update_lag_ratio"] = (df["days_since_last_update"] / df["content_age_days"].replace(0, np.nan)).fillna(0)

df["stale_and_declining"].value_counts()

### Search opportunity
Where a page is leaving clicks on the table relative to demand (`opportunity_score`), and how much
of the available search visibility it's actually capturing (`visibility_gap`). Log versions are
added since both are heavily right-skewed.

In [ ]:
df["opportunity_score"] = df["search_volume"] * (1 - df["ctr"])
df["opportunity_score_log"] = np.log1p(df["opportunity_score"].clip(lower=0))

df["visibility_gap"] = df["impressions_90d"] / (df["search_volume"] + 1)
df["visibility_gap_log"] = np.log1p(df["visibility_gap"])

df["low_ctr_high_volume"] = (
    (df["ctr"] < df["ctr"].median()) & (df["search_volume"] > df["search_volume"].median())
).astype(int)

### Content thinness
Pages in the bottom quartile of word count are flagged as "thin," and engagement is normalized
per word to control for length when comparing engagement across pages.

In [ ]:
df["thin_content_flag"] = (df["word_count"] < df["word_count"].quantile(0.25)).astype(int)
df["engagement_per_word"] = df["engagement_rate"] / (df["word_count"] + 1)

### Combination flags
Pages that are simultaneously declining and thin, or stale/declining and thin, are likely the
strongest refresh candidates — these combination flags surface that directly as features.

In [ ]:
df["decline_and_thin"] = (df["traffic_change_pct"] < -0.10).astype(int) * df["thin_content_flag"]
df["stale_decline_thin"] = df["stale_and_declining"] * df["thin_content_flag"]

### Validate the new features
Check for any nulls or infinities introduced during feature engineering before moving on.

In [ ]:
new_cols = [
    "traffic_change_pct", "impressions_change_pct", "sessions_change_pct",
    "clicks_per_active_day", "impressions_per_active_day", "impression_coverage", "session_coverage",
    "engaged_session_rate", "scroll_events_per_session", "pageviews_per_user",
    "ai_session_share", "stale_and_declining", "stale_decline_thin", "update_lag_ratio",
    "opportunity_score", "opportunity_score_log", "visibility_gap", "visibility_gap_log",
    "low_ctr_high_volume", "thin_content_flag", "engagement_per_word", "decline_and_thin",
]

print(df[new_cols].isnull().sum().sum(), "total nulls")
print(np.isinf(df[new_cols].select_dtypes(include=[np.number])).sum().sum(), "total infs")
df[new_cols].describe()

## 8. Feature Redundancy Check

`visibility_gap` and its log transform are highly correlated, as expected — the log version is
kept for modeling since it's less skewed, and the raw version is dropped later to avoid feeding
the model two versions of the same signal.

In [ ]:
print(df[["visibility_gap", "visibility_gap_log"]].corr(method="spearman"))
print(df[["impressions_per_active_day", "impression_coverage"]].corr())

## 9. Label Definition

The target, `needs_refresh`, flags a page as a refresh priority if it is both **poorly ranked**
(`avg_position > 20`, i.e. off page 1–2) and **stale and declining** (per the feature built above).
This is deliberately stricter than a simpler "poorly ranked and trending down" rule, since it
requires an actual staleness signal (time since last update) rather than trend direction alone —
which cuts down on false positives from pages that are simply seasonal.

In [ ]:
df["needs_refresh"] = (
    (df["avg_position"] > 20) &
    (df["stale_and_declining"] == 1)
).astype(int)

df["needs_refresh"].value_counts(normalize=True)

## 10. Encoding and Leakage-Safe Feature Matrix

Low-cardinality categorical columns are one-hot encoded. The feature matrix then explicitly
excludes:
- identifiers (`content_id`, `client_id`),
- the label itself,
- and any column used to *construct* the label (`avg_position`, `stale_and_declining`,
  `stale_decline_thin`, `trend_direction`/`position_tier` and their dummies, `decline_and_thin`) —
  these would otherwise leak the answer directly into the model.

`visibility_gap` is also dropped in favor of its log-transformed version (see the correlation
check above).

In [ ]:
low_cardinality_cats = [
    "content_type", "main_intent", "provider_used", "model_used",
    "competition_level", "age_tier", "freshness_tier",
    "word_count_tier", "char_count_tier", "impression_tier",
    "position_tier", "trend_direction",
]
df_encoded = pd.get_dummies(
    df, columns=[c for c in low_cardinality_cats if c in df.columns], drop_first=True
)

drop_cols = [
    "content_id", "client_id",               # identifiers, not features
    "needs_refresh",                          # the label itself
    "avg_position",                           # used to build the label
    "stale_and_declining",                    # used to build the label
    "stale_decline_thin",                     # derived from stale_and_declining
    "decline_and_thin",                       # partially overlaps trend logic
    "visibility_gap",                         # redundant with visibility_gap_log
]
X = df_encoded.drop(columns=[c for c in drop_cols if c in df_encoded.columns])
y = df_encoded["needs_refresh"]

# Drop dummy columns derived from label-construction columns
leakage_dummy_prefixes = ("trend_direction_", "position_tier_")
X = X.drop(columns=[c for c in X.columns if c.startswith(leakage_dummy_prefixes)])

print(X.shape, y.shape)
y.value_counts(normalize=True)

## 11. Train/Test Split

Stratified to preserve the class balance of `needs_refresh` in both splits, since the positive
class is a minority.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Train shape:", X_train.shape, "Test shape:", X_test.shape)
print("Train label balance:\n", y_train.value_counts(normalize=True))
print("Test label balance:\n", y_test.value_counts(normalize=True))

## 12. Feature Scaling

Standardize numeric features for the logistic regression baseline (tree-based models don't need
this, but it doesn't hurt to keep a scaled copy around).

In [ ]:
from sklearn.preprocessing import StandardScaler

numeric_cols = X_train.select_dtypes(include=["int64", "float64"]).columns

scaler = StandardScaler()
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()
X_train_scaled[numeric_cols] = scaler.fit_transform(X_train[numeric_cols])
X_test_scaled[numeric_cols] = scaler.transform(X_test[numeric_cols])

X_train_scaled[numeric_cols].describe().loc[["mean", "std"]]

## 13. Baseline Models

Two baselines, both with class weighting to account for the class imbalance:
- **Logistic regression** on the scaled features, as an interpretable linear baseline.
- **Random forest**, which doesn't need scaling and can capture non-linear interactions.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

log_reg = LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42)
log_reg.fit(X_train_scaled, y_train)
y_pred_lr = log_reg.predict(X_test_scaled)
print("Logistic Regression:\n", classification_report(y_test, y_pred_lr))

rf = RandomForestClassifier(class_weight="balanced", random_state=42, n_estimators=200)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)
print("Random Forest:\n", classification_report(y_test, y_pred_rf))

## 14. Threshold Tuning

The default 0.5 decision threshold isn't necessarily right for an imbalanced prioritization task —
lowering it trades some precision for recall, which matters if the business cost of *missing* a
page that needs a refresh is higher than the cost of reviewing a false positive.

In [ ]:
probs_rf = rf.predict_proba(X_test)[:, 1]

for t in [0.5, 0.4, 0.3, 0.2]:
    y_pred_thresh = (probs_rf > t).astype(int)
    print(f"--- threshold = {t} ---")
    print(classification_report(y_test, y_pred_thresh))

## 15. Feature Importance

Which features the random forest actually relies on most.

In [ ]:
importances = pd.Series(rf.feature_importances_, index=X_train.columns).sort_values(ascending=False)
importances.head(20)

## 16. Gradient Boosting Comparison

A gradient boosting classifier as a further comparison point against the random forest baseline.

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier

gb = GradientBoostingClassifier(random_state=42)
gb.fit(X_train, y_train)
y_pred_gb = gb.predict(X_test)
print(classification_report(y_test, y_pred_gb))

## 17. Cross-Validation

A single train/test split can be noisy with this much class imbalance, so the random forest is
re-evaluated with 5-fold stratified cross-validation, and the chosen threshold (0.3) is validated
on out-of-fold predictions to avoid any optimistic bias from tuning on the test set.

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_score, cross_val_predict
from sklearn.metrics import f1_score

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_model = RandomForestClassifier(random_state=42, class_weight="balanced", n_estimators=200)

f1_scores = cross_val_score(cv_model, X, y, cv=skf, scoring="f1")
print("F1 per fold:", np.round(f1_scores, 3))
print("Mean F1:", round(f1_scores.mean(), 3))
print("Std F1:", round(f1_scores.std(), 3))

In [ ]:
# Out-of-fold probabilities, used to validate the chosen threshold without touching the test set
oof_probs = cross_val_predict(cv_model, X, y, cv=skf, method="predict_proba")[:, 1]

oof_preds_at_03 = (oof_probs >= 0.3).astype(int)
f1_at_03 = f1_score(y, oof_preds_at_03)
print("F1 at threshold 0.3 (cross-validated):", round(f1_at_03, 3))

In [ ]:
from sklearn.metrics import precision_recall_curve

precision, recall, thresholds = precision_recall_curve(y, oof_probs)

plt.figure(figsize=(7, 5))
plt.plot(thresholds, precision[:-1], label="Precision")
plt.plot(thresholds, recall[:-1], label="Recall")
plt.axvline(x=0.3, color="gray", linestyle="--", label="chosen threshold (0.3)")
plt.xlabel("Threshold")
plt.ylabel("Score")
plt.title("Precision-Recall vs Threshold")
plt.legend()
plt.show()

## 18. Summary

- The refresh-need label combines rank (`avg_position > 20`) with a staleness signal
  (`stale_and_declining`), which is stricter and less noisy than rank + trend alone.
- The feature set was built to be leakage-safe: every column used to construct the label was
  excluded from the model inputs.
- Random forest and gradient boosting both outperform the logistic regression baseline, which is
  expected given the non-linear interactions between staleness, engagement, and search opportunity
  features.
- A threshold of ~0.3 (rather than the default 0.5) improves recall at a modest precision cost,
  which is validated with cross-validated out-of-fold predictions rather than test-set tuning.

**Possible next steps:** hyperparameter tuning (grid/random search on the random forest and
gradient boosting models), testing an XGBoost/LightGBM model, and framing the business cost of
false negatives vs. false positives more precisely to justify the final threshold choice.